# Test 2 - Wine Quality

In [ ]:
import pandas as pd

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

In [ ]:
from models import KNN
from utils import metrics as mtr
from utils import preprocessing as prp
from utils import visuals as vis

In [ ]:
df = pd.read_csv('../data/raw/WineQT.csv')
df.head()

In [ ]:
Y = df['quality'].values
X = df.drop(['quality', 'Id'], axis=1).values

X_train, Y_train, X_test, Y_test = prp.train_test_split(X, Y)

In [ ]:
z_score = prp.normalize.z_score()

X_train = z_score.fit_transform(X_train)
X_test  = z_score.transform(X_test)

In [ ]:
vis.style(style='darkgrid')

# its 2 choose 11 = 55 plots, might take a while to render
vis.plot_feature_relationships(
    X_train, Y_train, 
    n_columns=2,
    feature_labels=['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol'], 
)

In [ ]:
voting_weight = lambda d: 1 / (d + 1e-8) ** 2

knn1 = KNN(k=1, voting_weight=voting_weight)
knn5 = KNN(k=5, voting_weight=voting_weight)
knn30 = KNN(k=30, voting_weight=voting_weight)

feature_weights=[0.5, 1.3, 0.9, 0.4, 0.6, 0.4, 0.8, 0.7, 0.5, 1.1, 1.5]

knn1.fit(X_train, Y_train, feature_weights=feature_weights)
knn5.fit(X_train, Y_train, feature_weights=feature_weights)
knn30.fit(X_train, Y_train, feature_weights=feature_weights)

In [ ]:
Y1_pred,  Y1_confs  = knn1.predict(X_test)
Y5_pred,  Y5_confs  = knn5.predict(X_test)
Y30_pred, Y30_confs = knn30.predict(X_test)

accuracies  = [round(mtr.accuracy_score(Y_test, y_pred), 4) for y_pred in [Y1_pred, Y5_pred, Y30_pred]]
confidences = [round(y_confs.mean(), 4) for y_confs in [Y1_confs, Y5_confs, Y30_confs]]

print("k=1:  " + str(accuracies[0]) + ", avg confidence: " + str(confidences[0]))
print("k=5:  " + str(accuracies[1]) + ", avg confidence: " + str(confidences[1]))
print("k=30: " + str(accuracies[2]) + ", avg confidence: " + str(confidences[2]))

In [ ]:
conf_mats = [mtr.confusion_matrix(Y_test, y_pred) for y_pred in [Y1_pred, Y5_pred, Y30_pred]]
Ks = [1, 5, 30]

vis.plot_confusion_matrices(
    conf_mats,
    ['Quality 3', 'Quality 4', 'Quality 5', 'Quality 6', 'Quality 7', 'Quality 8'],
    titling=lambda i: f"Confusion Matrix for K = {Ks[i]}"
)